# KPI 1 — Monthly Revenue Trend

**requirements.md (Sales & Revenue, KPI 1):** "Monthly Revenue Trend: Gross and net revenue (excluding cancelled orders) by month with MoM growth %."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_1_monthly_revenue_trend` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cell below once (or after any change to the fact/dim definitions).

**Definitions (state formulas):**

* **Grain:** one row per `year_month` (`dim_date`, `YYYYMM` int)
* **Gross revenue** = `SUM(line_total)` over ALL order statuses with `order_date_valid = true`
* **Net revenue** = `SUM(line_total)` over `status <> 'cancelled'` with `order_date_valid = true`
* Invalid dates (null / future `order_date`) are excluded from BOTH bases to prevent phantom 2027 months (rows stay in the fact, flagged by `order_date_valid`)
* **MoM growth %** = `ROUND(100 × (current − previous) / previous, 2)` — `NULL` for the first month (2024-10)

**Verified 2026-11-19:** 26 contiguous months 2024-10 → 2026-11; net total $28,333,498.48 (exact gold-audit match); gross (valid dates) $30,723,269.53; all 25 MoM values match an independent self-join recompute on both bases.

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_1_monthly_revenue_trend
TBLPROPERTIES (
  'comment' = 'KPI 1 - Monthly Revenue Trend (requirements.md: gross + net revenue excluding cancelled orders, by month, with MoM growth %). Grain: one row per year_month (dim_date, YYYYMM int). Gross = SUM(fact_order_items.line_total) over ALL order statuses with order_date_valid = true. Net = SUM(line_total) over status <> ''cancelled'' with order_date_valid = true. Invalid dates (null/future order_date) are excluded from BOTH bases to prevent phantom 2027 months (flagged in the fact; rows retained there). MoM growth % = ROUND(100 * (current - LAG(current)) / LAG(current), 2); NULL for the first month (2024-10).'
)
AS
WITH monthly AS (
  SELECT
    d.year_month,
    SUM(f.line_total)                                                    AS gross_revenue,
    SUM(CASE WHEN f.status <> 'cancelled' THEN f.line_total ELSE 0 END)  AS net_revenue
  FROM b2b_ecommerce.gold.fact_order_items f
  JOIN b2b_ecommerce.gold.dim_date        d ON f.date_sk = d.date_sk
  WHERE f.order_date_valid
  GROUP BY d.year_month
),
mom AS (
  SELECT
    year_month,
    gross_revenue,
    net_revenue,
    LAG(gross_revenue) OVER (ORDER BY year_month) AS prev_gross,
    LAG(net_revenue)   OVER (ORDER BY year_month) AS prev_net
  FROM monthly
)
SELECT
  year_month,
  gross_revenue,
  net_revenue,
  ROUND(100 * (gross_revenue - prev_gross) / prev_gross, 2) AS gross_mom_growth_pct,
  ROUND(100 * (net_revenue - prev_net) / prev_net, 2)       AS net_mom_growth_pct
FROM mom;

In [0]:
%sql
-- Verification: shape, totals, boundaries. Expected: 26 months 202410-202611,
-- SUM(net) = 28,333,498.48, SUM(gross) = 30,723,269.53, no months outside window, 1 NULL MoM row.
SELECT
  COUNT(*)            AS rows_months,
  MIN(year_month)     AS min_month,
  MAX(year_month)     AS max_month,
  SUM(gross_revenue)  AS sum_gross,
  SUM(net_revenue)    AS sum_net,
  SUM(CASE WHEN year_month NOT BETWEEN 202410 AND 202611 THEN 1 ELSE 0 END) AS months_outside_window,
  COUNT(*) - COUNT(gross_mom_growth_pct) AS expected_null_mom_rows
FROM b2b_ecommerce.gold.kpi_1_monthly_revenue_trend;

In [0]:
%sql
-- Verification: independently recompute EVERY MoM value via self-join (not LAG).
-- Expected: 25 pairs, gross_mom_match = 25, net_mom_match = 25.
WITH v AS (
  SELECT year_month, gross_revenue, net_revenue,
         gross_mom_growth_pct, net_mom_growth_pct,
         ROW_NUMBER() OVER (ORDER BY year_month) AS rn
  FROM b2b_ecommerce.gold.kpi_1_monthly_revenue_trend
)
SELECT
  COUNT(*) AS mom_pairs_checked,
  SUM(CASE WHEN a.gross_mom_growth_pct = ROUND(100*(a.gross_revenue - b.gross_revenue)/b.gross_revenue, 2) THEN 1 ELSE 0 END) AS gross_mom_match,
  SUM(CASE WHEN a.net_mom_growth_pct   = ROUND(100*(a.net_revenue   - b.net_revenue)/b.net_revenue, 2)         THEN 1 ELSE 0 END) AS net_mom_match
FROM v a
JOIN v b ON a.rn = b.rn + 1;

In [0]:
%sql
-- Full trend
SELECT * FROM b2b_ecommerce.gold.kpi_1_monthly_revenue_trend ORDER BY year_month;